# **Imports Config**

# **Updated U-Net with Train/Val/Test Dataloaders**

## **Key Updates Made:**
1. ✅ **Complete train/validation/test dataloader support** - The notebook now properly creates and handles all three dataloaders
2. ✅ **Fixed model architecture** - Added missing `ChannelAttention`, `SpatialAttention`, and `DAGConvBlock` components
3. ✅ **Improved dataset handling** - Enhanced dataset loading with flexible filename matching and index-based fallback
4. ✅ **Test set evaluation** - Added test set evaluation after training completion
5. ✅ **Better error handling** - Improved error messages and validation checks

## **Before Running:**
1. **Update the dataset path** in the configuration section (cell with `CONFIG['dataset_path']`)
2. **Ensure your dataset follows the expected structure:**
   ```
   your_dataset/
   ├── images/
   │   ├── image1.png
   │   ├── image2.jpg
   │   └── ...
   └── masks/
       ├── image1.png (or image1_mask.png)
       ├── image2.png (or image2_mask.png)
       └── ...
   ```
3. **Run all cells in order** - The notebook now has proper dependencies between cells

## **Dataset Split:**
- **Training: 70%** - Used for model training
- **Validation: 15%** - Used for model selection and early stopping
- **Test: 15%** - Used for final evaluation (optional)

**The notebook will automatically handle all three datasets and evaluate performance on each!**

In [ ]:
import os
import random
import warnings
from pathlib import Path
from typing import Dict, Tuple, Optional
import glob

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler

import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

In [ ]:
CONFIG = {
    # Dataset
    'dataset_path': '/kaggle/input/monuseg2018',  # Fallback to local        
    'input_size': (256, 256),
    'num_classes': 1,
    
    # Training
    'batch_size': 8,  # Per GPU
    'num_epochs': 30,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'grad_clip': 1.0,
    
    # Loss weights
    'alpha_bce': 0.5,
    'beta_dice': 0.5,
    'label_smoothing': 0.0,
    
    # Optimizer & Scheduler
    'optimizer': 'Adam',
    'momentum': 0.9,
    'scheduler': 'CosineAnnealingLR',
    'scheduler_params': {'T_max': 100, 'eta_min': 1e-6},
    
    # Regularization
    'dropout': 0.1,
    'use_mixed_precision': True,
    
    # Callbacks
    'early_stopping_patience': 15,
    'reduce_lr_patience': 7,
    'reduce_lr_factor': 0.5,
    
    # Logging
    'log_dir': './logs',
    'checkpoint_dir': './checkpoints',
    'save_predictions': True,
    'num_vis_samples': 4,
    
    # Reproducibility
    'seed': 42,
    'num_workers': 4,
    
    # Resume training
    'resume_from': None,  # Path to checkpoint or None
}

# ============================================================================
# Reproducibility
# ============================================================================

def seed_everything(seed: int):
    """Set seeds for reproducibility"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CONFIG['seed'])

# **Load Dataset**

## **Dataset Structure Requirements**

Your dataset should be organized as follows:
```
dataset_folder/
├── images/          # All input images
│   ├── image1.png   # Any naming convention
│   ├── img_001.jpg  # Mixed formats supported
│   └── ...
└── masks/           # Corresponding masks  
    ├── image1.png   # Same name as image (preferred)
    ├── img_001_mask.png  # Or with _mask suffix
    └── ...
```

**Supported Naming Conventions for Masks:**
- Same as image: `image1.png` → `image1.png`
- With suffix: `image1.png` → `image1_mask.png`, `image1_gt.png`, `image1_label.png`
- With prefix: `image1.png` → `mask_image1.png`, `gt_image1.png`
- **Index-based**: If no naming pattern works, files will be matched by alphabetical order

**Before running the cells below:**
1. Update `DATASET_PATH` in the next cell to point to your dataset folder
2. Run the cells to create train/validation/test data loaders
3. The code will automatically detect the naming pattern or fall back to index matching

In [ ]:
class getDataset(Dataset):
    """Dataset with augmentations"""
    
    def __init__(self, images, masks, transform=None):
        self.images = images
        self.masks = masks
        self.transform = transform
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        mask = self.masks[idx]
        
        # Ensure correct format
        if len(image.shape) == 2:
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        if len(mask.shape) == 3:
            mask = mask[:, :, 0]
        
        # Normalize mask to [0, 1]
        mask = (mask > 0).astype(np.float32)
        
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']
        
        return image, mask.unsqueeze(0)


def get_transforms(train: bool = True):
    """Get augmentation transforms"""
    if train:
        return A.Compose([
            A.Resize(*CONFIG['input_size']),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, 
                              rotate_limit=15, p=0.5),
            A.OneOf([
                A.GaussNoise(var_limit=(10.0, 50.0), p=1),
                A.GaussianBlur(blur_limit=(3, 7), p=1),
            ], p=0.3),
            A.OneOf([
                A.RandomBrightnessContrast(brightness_limit=0.2, 
                                          contrast_limit=0.2, p=1),
                A.HueSaturationValue(hue_shift_limit=20, 
                                    sat_shift_limit=30, 
                                    val_shift_limit=20, p=1),
            ], p=0.3),
            A.Normalize(mean=[0.485, 0.456, 0.406], 
                       std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(*CONFIG['input_size']),
            A.Normalize(mean=[0.485, 0.456, 0.406], 
                       std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])


In [ ]:
import glob
from sklearn.model_selection import train_test_split
import os
from pathlib import Path

def find_matching_masks(image_paths, masks_dir):
    """
    Find matching mask files for images using various naming conventions
    """
    valid_pairs = []
    unmatched_images = []
    
    # Common image and mask file extensions
    mask_extensions = ['.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp']
    
    for img_path in image_paths:
        img_name = Path(img_path).stem
        mask_found = False
        
        # Try different naming conventions for masks
        possible_mask_names = [
            img_name,                    # exact same name
            f"{img_name}_mask",          # image_name + _mask
            f"{img_name}_gt",            # image_name + _gt (ground truth)
            f"{img_name}_label",         # image_name + _label
            f"{img_name}_anno",          # image_name + _anno (annotation)
            f"mask_{img_name}",          # mask_ + image_name
            f"gt_{img_name}",            # gt_ + image_name
            f"label_{img_name}",         # label_ + image_name
        ]
        
        # Try each possible mask name with each extension
        for mask_name in possible_mask_names:
            for ext in mask_extensions:
                mask_path = Path(masks_dir) / f"{mask_name}{ext}"
                if mask_path.exists():
                    valid_pairs.append((img_path, str(mask_path)))
                    mask_found = True
                    break
            if mask_found:
                break
        
        if not mask_found:
            unmatched_images.append(img_path)
    
    return valid_pairs, unmatched_images

def load_dataset_from_folders(data_path, train_split=0.7, val_split=0.15, test_split=0.15, 
                             match_by_index=False):
    """
    Load dataset from images and masks directories and split into train/val/test
    
    Args:
        data_path: Path to directory containing 'images' and 'masks' folders
        train_split: Proportion of data for training (default: 0.7)
        val_split: Proportion of data for validation (default: 0.15) 
        test_split: Proportion of data for testing (default: 0.15)
        match_by_index: If True, match images and masks by alphabetical order index
    
    Returns:
        Dictionary containing train, val, and test image and mask paths
    """
    data_path = Path(data_path)
    
    # Get directories
    images_dir = data_path / 'images'
    masks_dir = data_path / 'masks'
    
    if not images_dir.exists():
        raise FileNotFoundError(f"Images directory not found: {images_dir}")
    if not masks_dir.exists():
        raise FileNotFoundError(f"Masks directory not found: {masks_dir}")
    
    # Get all image files (support common formats)
    image_extensions = ['*.png', '*.jpg', '*.jpeg', '*.tif', '*.tiff', '*.bmp']
    image_paths = []
    
    for ext in image_extensions:
        image_paths.extend(glob.glob(str(images_dir / ext)))
    
    image_paths = sorted(image_paths)  # Sort for consistent ordering
    print(f"Found {len(image_paths)} images in {images_dir}")
    
    if match_by_index:
        # Match by alphabetical order (index-based matching)
        mask_extensions = ['*.png', '*.jpg', '*.jpeg', '*.tif', '*.tiff', '*.bmp']
        mask_paths = []
        
        for ext in mask_extensions:
            mask_paths.extend(glob.glob(str(masks_dir / ext)))
        
        mask_paths = sorted(mask_paths)
        print(f"Found {len(mask_paths)} masks in {masks_dir}")
        
        if len(image_paths) != len(mask_paths):
            print(f"Warning: Number of images ({len(image_paths)}) != number of masks ({len(mask_paths)})")
            min_len = min(len(image_paths), len(mask_paths))
            image_paths = image_paths[:min_len]
            mask_paths = mask_paths[:min_len]
            print(f"Using first {min_len} files from each directory")
        
        valid_pairs = list(zip(image_paths, mask_paths))
        unmatched_images = []
    else:
        # Match by filename patterns
        valid_pairs, unmatched_images = find_matching_masks(image_paths, masks_dir)
    
    print(f"Found {len(valid_pairs)} valid image-mask pairs")
    
    if unmatched_images:
        print(f"Warning: {len(unmatched_images)} images have no matching masks:")
        for img in unmatched_images[:5]:  # Show first 5
            print(f"  - {Path(img).name}")
        if len(unmatched_images) > 5:
            print(f"  ... and {len(unmatched_images) - 5} more")
    
    if len(valid_pairs) == 0:
        raise ValueError("No valid image-mask pairs found! Try setting match_by_index=True")
    
    # Validate splits sum to 1
    assert abs(train_split + val_split + test_split - 1.0) < 1e-6, "Splits must sum to 1.0"
    
    # Split dataset
    # First split: separate test set
    if test_split > 0:
        train_val_pairs, test_pairs = train_test_split(
            valid_pairs, 
            test_size=test_split, 
            random_state=CONFIG['seed'],
            shuffle=True
        )
    else:
        train_val_pairs = valid_pairs
        test_pairs = []
    
    # Second split: separate train and validation
    if val_split > 0:
        val_size_adjusted = val_split / (train_split + val_split)
        train_pairs, val_pairs = train_test_split(
            train_val_pairs,
            test_size=val_size_adjusted,
            random_state=CONFIG['seed'],
            shuffle=True
        )
    else:
        train_pairs = train_val_pairs
        val_pairs = []
    
    print(f"Dataset split: Train={len(train_pairs)}, Val={len(val_pairs)}, Test={len(test_pairs)}")
    
    return {
        'train': {'images': [pair[0] for pair in train_pairs], 
                 'masks': [pair[1] for pair in train_pairs]},
        'val': {'images': [pair[0] for pair in val_pairs], 
               'masks': [pair[1] for pair in val_pairs]} if val_pairs else {'images': [], 'masks': []},
        'test': {'images': [pair[0] for pair in test_pairs], 
                'masks': [pair[1] for pair in test_pairs]} if test_pairs else {'images': [], 'masks': []}
    }


def load_images_and_masks(image_paths, mask_paths):
    """Load images and masks from file paths"""
    images = []
    masks = []
    
    print("Loading images and masks...")
    for img_path, mask_path in tqdm(zip(image_paths, mask_paths), total=len(image_paths)):
        # Load image
        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Could not load image {img_path}")
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Load mask
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            print(f"Warning: Could not load mask {mask_path}")
            continue
            
        images.append(img)
        masks.append(mask)
    
    return np.array(images), np.array(masks)


def create_data_loaders(data_path, batch_size=None, num_workers=None, match_by_index=False):
    """
    Create train, validation, and test data loaders
    
    Args:
        data_path: Path to directory containing 'images' and 'masks' folders
        batch_size: Batch size (defaults to CONFIG value)
        num_workers: Number of workers (defaults to CONFIG value)
        match_by_index: If True, match images and masks by alphabetical order
    
    Returns:
        Dictionary containing train_loader, val_loader, test_loader
    """
    if batch_size is None:
        batch_size = CONFIG['batch_size']
    if num_workers is None:
        num_workers = CONFIG['num_workers']
    
    # Load and split dataset
    dataset_splits = load_dataset_from_folders(data_path, match_by_index=match_by_index)
    
    # Load images and masks for each split
    datasets = {}
    loaders = {}
    
    for split in ['train', 'val', 'test']:
        if len(dataset_splits[split]['images']) > 0:
            print(f"\nLoading {split} data...")
            images, masks = load_images_and_masks(
                dataset_splits[split]['images'], 
                dataset_splits[split]['masks']
            )
            
            # Get transforms
            transform = get_transforms(train=(split == 'train'))
            
            # Create dataset
            dataset = getDataset(images, masks, transform=transform)
            datasets[split] = dataset
            
            # Create data loader
            shuffle = (split == 'train')
            loader = DataLoader(
                dataset,
                batch_size=batch_size,
                shuffle=shuffle,
                num_workers=num_workers,
                pin_memory=torch.cuda.is_available(),
                drop_last=(split == 'train')
            )
            loaders[f'{split}_loader'] = loader
            
            print(f"{split.capitalize()} dataset: {len(dataset)} samples")
        else:
            print(f"No {split} data available")
            loaders[f'{split}_loader'] = None
    
    return loaders

# Example usage and verification function
def verify_dataset_structure(data_path, sample_size=5):
    """
    Verify and display information about the dataset structure
    """
    data_path = Path(data_path)
    images_dir = data_path / 'images'
    masks_dir = data_path / 'masks'
    
    print("=== Dataset Structure Verification ===")
    print(f"Dataset path: {data_path}")
    print(f"Images directory: {images_dir} (exists: {images_dir.exists()})")
    print(f"Masks directory: {masks_dir} (exists: {masks_dir.exists()})")
    
    if images_dir.exists():
        image_files = []
        for ext in ['*.png', '*.jpg', '*.jpeg', '*.tif', '*.tiff', '*.bmp']:
            image_files.extend(glob.glob(str(images_dir / ext)))
        print(f"Total images: {len(image_files)}")
        
        if image_files and sample_size > 0:
            print(f"Sample image files:")
            for i, img in enumerate(sorted(image_files)[:sample_size]):
                print(f"  {i+1}. {Path(img).name}")
    
    if masks_dir.exists():
        mask_files = []
        for ext in ['*.png', '*.jpg', '*.jpeg', '*.tif', '*.tiff', '*.bmp']:
            mask_files.extend(glob.glob(str(masks_dir / ext)))
        print(f"Total masks: {len(mask_files)}")
        
        if mask_files and sample_size > 0:
            print(f"Sample mask files:")
            for i, mask in enumerate(sorted(mask_files)[:sample_size]):
                print(f"  {i+1}. {Path(mask).name}")

In [ ]:
# ============================================================================
# Dataset Loading and Data Loaders Creation
# ============================================================================

# Set your dataset path here
DATASET_PATH = CONFIG['dataset_path']  # Update this to your actual dataset path

# Option 1: Try to match images and masks by filename patterns (recommended first)
print("=== Attempting filename-based matching ===")
try:
    data_loaders = create_data_loaders(DATASET_PATH, match_by_index=False)
    print("✅ Successfully created data loaders using filename matching!")
except Exception as e:
    print(f"❌ Filename matching failed: {e}")
    print("\n=== Trying index-based matching ===")
    # Option 2: Fallback to index-based matching (alphabetical order)
    data_loaders = create_data_loaders(DATASET_PATH, match_by_index=True)
    print("✅ Successfully created data loaders using index-based matching!")

# Extract individual loaders
train_loader = data_loaders['train_loader']
val_loader = data_loaders['val_loader'] 
test_loader = data_loaders['test_loader']

print(f"\n=== Data Loaders Summary ===")
print(f"Train loader: {len(train_loader) if train_loader else 0} batches")
print(f"Val loader: {len(val_loader) if val_loader else 0} batches") 
print(f"Test loader: {len(test_loader) if test_loader else 0} batches")

# Verify first batch
if train_loader:
    print("\n=== Verifying first training batch ===")
    sample_batch = next(iter(train_loader))
    images, masks = sample_batch
    print(f"Batch images shape: {images.shape}")
    print(f"Batch masks shape: {masks.shape}")
    print(f"Images dtype: {images.dtype}, range: [{images.min():.3f}, {images.max():.3f}]")
    print(f"Masks dtype: {masks.dtype}, range: [{masks.min():.3f}, {masks.max():.3f}]")
    print(f"Unique mask values: {torch.unique(masks)}")

# Optional: Verify dataset structure before loading
# Uncomment the line below to see your dataset structure
# verify_dataset_structure(DATASET_PATH, sample_size=10)

In [ ]:
def visualize_samples(data_loader, num_samples=4, figsize=(15, 10)):
    """
    Visualize image-mask pairs from a data loader
    """
    if data_loader is None:
        print("Data loader is None")
        return
    
    # Get a batch
    images, masks = next(iter(data_loader))
    
    # Convert to numpy for visualization
    images = images[:num_samples].cpu().numpy()
    masks = masks[:num_samples].cpu().numpy()
    
    # Denormalize images (reverse ImageNet normalization)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    
    fig, axes = plt.subplots(2, num_samples, figsize=figsize)
    
    for i in range(num_samples):
        # Denormalize and convert image
        img = images[i].transpose(1, 2, 0)  # CHW to HWC
        img = img * std + mean
        img = np.clip(img, 0, 1)
        
        # Get mask
        mask = masks[i, 0]  # Remove channel dimension
        
        # Plot image
        axes[0, i].imshow(img)
        axes[0, i].set_title(f'Image {i+1}')
        axes[0, i].axis('off')
        
        # Plot mask
        axes[1, i].imshow(mask, cmap='gray')
        axes[1, i].set_title(f'Mask {i+1}')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize samples from each dataset split
print("=== Visualizing Dataset Samples ===")
if train_loader:
    print("Training samples:")
    visualize_samples(train_loader, num_samples=4)

if val_loader:
    print("Validation samples:")
    visualize_samples(val_loader, num_samples=4)

if test_loader:
    print("Test samples:")
    visualize_samples(test_loader, num_samples=4)

# **Create Data Loaders**

In [ ]:
# Update the dataset path in CONFIG to point to your data directory
CONFIG['dataset_path'] = '/path/to/your/dataset'  # Update this path!

# Create the data loaders using the comprehensive function
try:
    data_loaders = create_data_loaders(CONFIG['dataset_path'], match_by_index=False)
    
    # Extract individual loaders
    train_loader = data_loaders['train_loader']
    val_loader = data_loaders['val_loader'] 
    test_loader = data_loaders['test_loader']
    
    print("\n" + "="*50)
    print("DATA LOADERS CREATED SUCCESSFULLY!")
    print("="*50)
    print(f"Train batches: {len(train_loader) if train_loader else 0}")
    print(f"Validation batches: {len(val_loader) if val_loader else 0}")
    print(f"Test batches: {len(test_loader) if test_loader else 0}")
    print(f"Batch size: {CONFIG['batch_size']}")
    
    # Test loading a batch from train loader
    if train_loader:
        train_batch = next(iter(train_loader))
        images, masks = train_batch
        print(f"\nBatch shapes:")
        print(f"Images: {images.shape}")  # Should be [batch_size, 3, height, width]
        print(f"Masks: {masks.shape}")    # Should be [batch_size, 1, height, width]
        print(f"Images dtype: {images.dtype}")
        print(f"Masks dtype: {masks.dtype}")
        print(f"Images range: [{images.min():.3f}, {images.max():.3f}]")
        print(f"Masks range: [{masks.min():.3f}, {masks.max():.3f}]")
    else:
        print("Warning: No training data available!")
    
except Exception as e:
    print(f"Error creating data loaders: {e}")
    print("Please check your dataset path and directory structure.")
    print("Expected structure:")
    print("dataset_path/")
    print("├── images/")
    print("│   ├── image1.png")
    print("│   ├── image2.png")
    print("│   └── ...")
    print("└── masks/")
    print("    ├── image1.png (or image1_mask.png)")
    print("    ├── image2.png (or image2_mask.png)")
    print("    └── ...")

# **Model**

In [ ]:
# ============================================================================
# Model Components
# ============================================================================

class ChannelAttention(nn.Module):
    """Channel Attention Module"""
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        
        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)


class SpatialAttention(nn.Module):
    """Spatial Attention Module"""
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        out = self.conv(out)
        return self.sigmoid(out)


class DAGConvBlock(nn.Module):
    """Dual Attention Guided Convolution Block"""
    def __init__(self, in_channels, out_channels, dropout=0.1):
        super().__init__()
        
        # Main convolution path
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout)
        )
        
        # Attention modules
        self.ca = ChannelAttention(out_channels)
        self.sa = SpatialAttention()
        
        # Skip connection projection if channels don't match
        self.skip = nn.Sequential()
        if in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        identity = self.skip(x)
        
        out = self.conv1(x)
        out = self.conv2(out)
        
        # Apply channel attention
        ca_weight = self.ca(out)
        out = out * ca_weight
        
        # Apply spatial attention
        sa_weight = self.sa(out)
        out = out * sa_weight
        
        # Add skip connection
        out = out + identity
        
        return out


class TransBlock(nn.Module):
    """
    Cascaded Multi-Scale Structure Block

    Now accepts separate channel counts:
      - in_deep_channels : channels of the deep feature (before projection)
      - in_shallow_channels : channels of the shallow feature (target channel)
    Projects the upsampled deep feature to in_shallow_channels with 1x1 conv,
    then adds to shallow features.
    """
    def __init__(self, in_deep_channels, in_shallow_channels):
        super().__init__()
        # project the upsampled deep feature to match shallow channels
        self.proj = nn.Sequential(
            nn.Conv2d(in_deep_channels, in_shallow_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_shallow_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, f_deep, f_shallow):
        # Upsample deep features to spatial size of shallow
        f_deep_up = F.interpolate(f_deep, size=f_shallow.shape[2:], mode='bilinear', align_corners=False)
        # Project channels to match shallow
        f_deep_up = self.proj(f_deep_up)
        # Residual-style addition (f_shallow is preserved)
        return f_shallow + f_deep_up


# ============================================================================
# Complete U-Net Model with Dual Attention
# ============================================================================

class DualAttentionUNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=1, base_channels=64, dropout=0.1):
        super().__init__()

        # Initial convolution
        self.init_conv = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, base_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
        )

        # Encoder with DAGConv blocks
        self.enc1 = DAGConvBlock(base_channels, base_channels * 2, dropout)
        self.enc2 = DAGConvBlock(base_channels * 2, base_channels * 4, dropout)
        self.enc3 = DAGConvBlock(base_channels * 4, base_channels * 8, dropout)
        self.enc4 = DAGConvBlock(base_channels * 8, base_channels * 16, dropout)

        self.pool = nn.MaxPool2d(2)

        # Cascaded multi-scale structure (Trans_Blocks)
        # Note: TransBlock now needs deep_channels and shallow_channels
        self.trans4 = TransBlock(in_deep_channels=base_channels * 16, in_shallow_channels=base_channels * 8)
        self.trans3 = TransBlock(in_deep_channels=base_channels * 8,  in_shallow_channels=base_channels * 4)
        self.trans2 = TransBlock(in_deep_channels=base_channels * 4,  in_shallow_channels=base_channels * 2)
        self.trans1 = TransBlock(in_deep_channels=base_channels * 2,  in_shallow_channels=base_channels)

        # Decoder
        self.up1 = nn.ConvTranspose2d(base_channels * 16, base_channels * 8, 2, stride=2)
        self.dec1 = DAGConvBlock(base_channels * 16, base_channels * 8, dropout)

        self.up2 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 2, stride=2)
        self.dec2 = DAGConvBlock(base_channels * 8, base_channels * 4, dropout)

        self.up3 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 2, stride=2)
        self.dec3 = DAGConvBlock(base_channels * 4, base_channels * 2, dropout)

        self.up4 = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, stride=2)
        self.dec4 = DAGConvBlock(base_channels * 2, base_channels, dropout)

        # Output
        self.out_conv = nn.Sequential(
            nn.Conv2d(base_channels, num_classes, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Initial features
        x0 = self.init_conv(x)  # base_channels

        # Encoder path
        x1 = self.pool(x0)
        x1 = self.enc1(x1)  # base_channels * 2

        x2 = self.pool(x1)
        x2 = self.enc2(x2)  # base_channels * 4

        x3 = self.pool(x2)
        x3 = self.enc3(x3)  # base_channels * 8

        x4 = self.pool(x3)
        x4 = self.enc4(x4)  # base_channels * 16 (bottleneck)

        # Cascaded multi-scale structure (bottom-up)
        x3_ms = self.trans4(x4, x3)   # returns channels = base_channels*8
        x2_ms = self.trans3(x3_ms, x2)  # returns channels = base_channels*4
        x1_ms = self.trans2(x2_ms, x1)  # returns channels = base_channels*2
        x0_ms = self.trans1(x1_ms, x0)  # returns channels = base_channels

        # Decoder path with multi-scale skip connections
        d1 = self.up1(x4)                       # channels = base*8
        d1 = torch.cat([d1, x3_ms], dim=1)     # channels = base*16
        d1 = self.dec1(d1)

        d2 = self.up2(d1)                       # channels = base*4
        d2 = torch.cat([d2, x2_ms], dim=1)     # channels = base*8
        d2 = self.dec2(d2)

        d3 = self.up3(d2)                       # channels = base*2
        d3 = torch.cat([d3, x1_ms], dim=1)     # channels = base*4
        d3 = self.dec3(d3)

        d4 = self.up4(d3)                       # channels = base
        d4 = torch.cat([d4, x0_ms], dim=1)     # channels = base*2
        d4 = self.dec4(d4)

        # Output
        out = self.out_conv(d4)

        return out

# Utility Func

In [ ]:
class DiceLoss(nn.Module):
    """Dice Loss for segmentation"""
    
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target):
        pred = pred.contiguous().view(-1)
        target = target.contiguous().view(-1)
        
        intersection = (pred * target).sum()
        dice = (2. * intersection + self.smooth) / \
               (pred.sum() + target.sum() + self.smooth)
        
        return 1 - dice


class CombinedLoss(nn.Module):
    """Combined BCE + Dice Loss"""
    
    def __init__(self, alpha=0.5, beta=0.5, label_smoothing=0.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.bce =  nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.label_smoothing = label_smoothing
    
    def forward(self, pred, target):
        if self.label_smoothing > 0:
            target = target * (1 - self.label_smoothing) + \
                    self.label_smoothing * 0.5
        
        bce_loss = self.bce(pred, target)
        dice_loss = self.dice(pred, target)
        
        return self.alpha * bce_loss + self.beta * dice_loss


# ============================================================================
# Metrics
# ============================================================================

def dice_coefficient(pred, target, threshold=0.5, smooth=1e-6):
    """Calculate Dice coefficient"""
    pred = (pred > threshold).float()
    target = (target > threshold).float()
    
    intersection = (pred * target).sum()
    dice = (2. * intersection + smooth) / \
           (pred.sum() + target.sum() + smooth)
    
    return dice.item()


def iou_score(pred, target, threshold=0.5, smooth=1e-6):
    """Calculate IoU (Intersection over Union)"""
    pred = (pred > threshold).float()
    target = (target > threshold).float()
    
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    iou = (intersection + smooth) / (union + smooth)
    
    return iou.item()


In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler, device, epoch):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0
    
    pbar = tqdm(loader, desc=f'Epoch {epoch} [Train]')
    for images, masks in pbar:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        
        # Mixed precision training
        with autocast(enabled=CONFIG['use_mixed_precision']):
            outputs = model(images)
            loss = criterion(outputs, masks)
        
        scaler.scale(loss).backward()
        
        # Gradient clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 
                                       CONFIG['grad_clip'])
        
        scaler.step(optimizer)
        scaler.update()
        
        # Metrics
        dice = dice_coefficient(outputs.detach(), masks.detach())
        iou = iou_score(outputs.detach(), masks.detach())
        
        running_loss += loss.item()
        running_dice += dice
        running_iou += iou
        
        pbar.set_postfix({'loss': loss.item(), 'dice': dice, 'iou': iou})
    
    n = len(loader)
    return running_loss / n, running_dice / n, running_iou / n


def validate_epoch(model, loader, criterion, device, epoch):
    """Validate for one epoch"""
    model.eval()
    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0
    
    pbar = tqdm(loader, desc=f'Epoch {epoch} [Val]')
    with torch.no_grad():
        for images, masks in pbar:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            dice = dice_coefficient(outputs, masks)
            iou = iou_score(outputs, masks)
            
            running_loss += loss.item()
            running_dice += dice
            running_iou += iou
            
            pbar.set_postfix({'loss': loss.item(), 'dice': dice, 'iou': iou})
    
    n = len(loader)
    return running_loss / n, running_dice / n, running_iou / n



In [ ]:
def save_predictions(model, loader, device, save_dir, epoch, num_samples=4):
    """Save prediction visualizations"""
    model.eval()
    save_path = Path(save_dir) / f'epoch_{epoch}'
    save_path.mkdir(parents=True, exist_ok=True)
    
    images, masks = next(iter(loader))
    images = images[:num_samples].to(device)
    masks = masks[:num_samples]
    
    with torch.no_grad():
        preds = model(images)
    
    images = images.cpu()
    preds = preds.cpu()
    
    # Denormalize images
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    images = images * std + mean
    images = torch.clamp(images, 0, 1)
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(num_samples):
        img = images[i].permute(1, 2, 0).numpy()
        mask = masks[i, 0].numpy()
        pred = preds[i, 0].numpy()
        
        axes[i, 0].imshow(img)
        axes[i, 0].set_title('Image')
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(mask, cmap='gray')
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(pred, cmap='gray')
        axes[i, 2].set_title(f'Prediction (Dice: {dice_coefficient(preds[i:i+1], masks[i:i+1]):.4f})')
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig(save_path / 'predictions.png', dpi=150, bbox_inches='tight')
    plt.close()


In [ ]:
def inference(model, image_path, device, threshold=0.5):
    """
    Run inference on a single image
    
    Args:
        model: Trained model
        image_path: Path to input image
        device: Computing device
        threshold: Prediction threshold
    
    Returns:
        pred_mask: Predicted binary mask
    """
    model.eval()
    
    # Load and preprocess image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    transform = get_transforms(train=False)
    transformed = transform(image=image)
    image_tensor = transformed['image'].unsqueeze(0).to(device)
    
    # Predict
    with torch.no_grad():
        pred = model(image_tensor)
    
    pred_mask = (pred.squeeze().cpu().numpy() > threshold).astype(np.uint8) * 255
    
    return pred_mask

# Dataloaders

In [ ]:
# Create directories
os.makedirs(CONFIG['log_dir'], exist_ok=True)
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)

# Ensure dataloaders exist (they should have been created in the previous cell)
if 'train_loader' not in globals() or train_loader is None:
    print("WARNING: train_loader not found! Please run the data loader creation cell first.")
    print("Creating dataloaders with default dataset path...")
    
    # Try to create dataloaders if they don't exist
    try:
        data_loaders = create_data_loaders(CONFIG['dataset_path'], match_by_index=False)
        train_loader = data_loaders['train_loader']
        val_loader = data_loaders['val_loader'] 
        test_loader = data_loaders['test_loader']
        
        if train_loader is None:
            raise ValueError("No training data available")
            
    except Exception as e:
        print(f"Failed to create dataloaders: {e}")
        print("Please check your dataset path and structure before training.")
        
print(f"\nDataloader Summary:")
print(f"Train batches: {len(train_loader) if train_loader else 0}")
print(f"Validation batches: {len(val_loader) if val_loader else 0}")
print(f"Test batches: {len(test_loader) if test_loader else 0}")

# Verify loaders are ready for training
if train_loader is None:
    raise ValueError("Training loader is required but not available!")
if val_loader is None:
    print("WARNING: No validation loader - training will proceed without validation")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DualAttentionUNet(
    in_channels=3,
    num_classes=CONFIG['num_classes'],
    base_channels=64,
    dropout=CONFIG['dropout']
)

# Multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
    model = nn.DataParallel(model)

model = model.to(device)

# Loss and optimizer
criterion = CombinedLoss(
    alpha=CONFIG['alpha_bce'],
    beta=CONFIG['beta_dice'],
    label_smoothing=CONFIG['label_smoothing']
)

In [ ]:
if CONFIG['optimizer'] == 'Adam':
    optimizer = optim.Adam(
        model.parameters(),
        lr=CONFIG['learning_rate'],
        weight_decay=CONFIG['weight_decay']
    )
elif CONFIG['optimizer'] == 'SGD':
    optimizer = optim.SGD(
        model.parameters(),
        lr=CONFIG['learning_rate'],
        momentum=CONFIG['momentum'],
        weight_decay=CONFIG['weight_decay']
    )
if CONFIG['scheduler'] == 'CosineAnnealingLR':
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, **CONFIG['scheduler_params']
    )
elif CONFIG['scheduler'] == 'ReduceLROnPlateau':
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=CONFIG['reduce_lr_factor'],
        patience=CONFIG['reduce_lr_patience'],
        verbose=True
    )

In [ ]:
# Mixed precision scaler
scaler = GradScaler(enabled=CONFIG['use_mixed_precision'])
# TensorBoard writer
writer = SummaryWriter(CONFIG['log_dir'])
# Training history
history = {
    'train_loss': [], 'train_dice': [], 'train_iou': [],
    'val_loss': [], 'val_dice': [], 'val_iou': []
}

# Resume from checkpoint
start_epoch = 0
best_dice = 0.0
early_stop_counter = 0

In [ ]:
if CONFIG['resume_from'] is not None and os.path.exists(CONFIG['resume_from']):
    print(f"Resuming from checkpoint: {CONFIG['resume_from']}")
    checkpoint = torch.load(CONFIG['resume_from'], map_location=device)
    
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint['model_state_dict'])
    
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_dice = checkpoint['best_dice']
    history = checkpoint['history']
    
    print(f"Resumed from epoch {start_epoch}, best dice: {best_dice:.4f}")

# Training loop
print(f"\n{'='*60}")
print(f"Starting training on {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")
print(f"{'='*60}\n")

# Train

In [ ]:
for epoch in range(start_epoch, CONFIG['num_epochs']):
    # Train
    train_loss, train_dice, train_iou = train_epoch(
        model, train_loader, criterion, optimizer, scaler, device, epoch
    )
    
    # Validate
    val_loss, val_dice, val_iou = validate_epoch(
        model, val_loader, criterion, device, epoch
    )
    
    # Update learning rate
    if CONFIG['scheduler'] == 'CosineAnnealingLR':
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]
    elif CONFIG['scheduler'] == 'ReduceLROnPlateau':
        scheduler.step(val_dice)
        current_lr = optimizer.param_groups[0]['lr']
    
    # Store history
    history['train_loss'].append(train_loss)
    history['train_dice'].append(train_dice)
    history['train_iou'].append(train_iou)
    history['val_loss'].append(val_loss)
    history['val_dice'].append(val_dice)
    history['val_iou'].append(val_iou)
    
    # TensorBoard logging
    writer.add_scalar('Loss/train', train_loss, epoch)
    writer.add_scalar('Loss/val', val_loss, epoch)
    writer.add_scalar('Dice/train', train_dice, epoch)
    writer.add_scalar('Dice/val', val_dice, epoch)
    writer.add_scalar('IoU/train', train_iou, epoch)
    writer.add_scalar('IoU/val', val_iou, epoch)
    writer.add_scalar('LearningRate', current_lr, epoch)
    
    # Print epoch summary
    print(f"\nEpoch {epoch} Summary:")
    print(f"  Train - Loss: {train_loss:.4f}, Dice: {train_dice:.4f}, IoU: {train_iou:.4f}")
    print(f"  Val   - Loss: {val_loss:.4f}, Dice: {val_dice:.4f}, IoU: {val_iou:.4f}")
    print(f"  LR: {current_lr:.6f}")
    
    # Save predictions
    if CONFIG['save_predictions'] and (epoch % 5 == 0 or epoch == CONFIG['num_epochs'] - 1):
        save_predictions(model, val_loader, device, 
                        os.path.join(CONFIG['log_dir'], 'predictions'),
                        epoch, CONFIG['num_vis_samples'])
    
    # Model checkpoint - save best model
    if val_dice > best_dice:
        best_dice = val_dice
        early_stop_counter = 0
        
        checkpoint_path = os.path.join(CONFIG['checkpoint_dir'], 
                                      'best_model.pth')
        
        save_dict = {
            'epoch': epoch,
            'model_state_dict': model.module.state_dict() if isinstance(model, nn.DataParallel) 
                               else model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_dice': best_dice,
            'val_loss': val_loss,
            'val_iou': val_iou,
            'history': history,
            'config': CONFIG
        }
        
        torch.save(save_dict, checkpoint_path)
        print(f"  ✓ Best model saved! Dice: {best_dice:.4f}")
    else:
        early_stop_counter += 1
    
    # Save latest checkpoint
    if epoch % 10 == 0:
        checkpoint_path = os.path.join(CONFIG['checkpoint_dir'], 
                                      f'checkpoint_epoch_{epoch}.pth')
        save_dict = {
            'epoch': epoch,
            'model_state_dict': model.module.state_dict() if isinstance(model, nn.DataParallel)
                               else model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_dice': best_dice,
            'val_loss': val_loss,
            'val_iou': val_iou,
            'history': history,
            'config': CONFIG
        }
        torch.save(save_dict, checkpoint_path)
        print(f"  Checkpoint saved at epoch {epoch}")
    
    # Early stopping
    if early_stop_counter >= CONFIG['early_stopping_patience']:
        print(f"\nEarly stopping triggered after {epoch + 1} epochs")
        print(f"Best Dice: {best_dice:.4f}")
        break


In [ ]:
final_checkpoint_path = os.path.join(CONFIG['checkpoint_dir'], 'final_model.pth')
save_dict = {
    'epoch': epoch,
    'model_state_dict': model.module.state_dict() if isinstance(model, nn.DataParallel)
                       else model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'best_dice': best_dice,
    'history': history,
    'config': CONFIG
}
torch.save(save_dict, final_checkpoint_path)

# Save training history to CSV
history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(CONFIG['log_dir'], 'training_history.csv'), 
                  index=False)

# Close TensorBoard writer
writer.close()

In [ ]:
print(f"\n{'='*60}")
print("Training completed!")
print(f"Best validation Dice: {best_dice:.4f}")
print(f"Final model saved to: {final_checkpoint_path}")
print(f"Best model saved to: {os.path.join(CONFIG['checkpoint_dir'], 'best_model.pth')}")
print(f"{'='*60}\n")

# Load best model for final evaluation
best_checkpoint = torch.load(os.path.join(CONFIG['checkpoint_dir'], 'best_model.pth'),map_location=device)

if isinstance(model, nn.DataParallel):
    model.module.load_state_dict(best_checkpoint['model_state_dict'])
else:
    model.load_state_dict(best_checkpoint['model_state_dict'])

# Final validation
print("Running final evaluation on validation set...")
final_loss, final_dice, final_iou = validate_epoch(
    model, val_loader, criterion, device, epoch='Final'
)

print(f"\nFinal Validation Metrics:")
print(f"  Loss: {final_loss:.4f}")
print(f"  Dice: {final_dice:.4f}")
print(f"  IoU:  {final_iou:.4f}")

# Test set evaluation if available
if test_loader is not None and len(test_loader) > 0:
    print("\nRunning evaluation on test set...")
    test_loss, test_dice, test_iou = validate_epoch(
        model, test_loader, criterion, device, epoch='Test'
    )
    
    print(f"\nFinal Test Metrics:")
    print(f"  Loss: {test_loss:.4f}")
    print(f"  Dice: {test_dice:.4f}")
    print(f"  IoU:  {test_iou:.4f}")
    
    # Save test predictions
    save_predictions(model, test_loader, device,
                    os.path.join(CONFIG['log_dir'], 'predictions'),
                    'test', num_samples=8)
else:
    print("No test set available for evaluation.")

# Plots

In [ ]:
# Save final predictions for all available datasets
if val_loader:
    save_predictions(model, val_loader, device,
                    os.path.join(CONFIG['log_dir'], 'predictions'),
                    'final_val', num_samples=8)

if test_loader:
    save_predictions(model, test_loader, device,
                    os.path.join(CONFIG['log_dir'], 'predictions'),
                    'final_test', num_samples=8)

# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Dice
axes[1].plot(history['train_dice'], label='Train', linewidth=2)
axes[1].plot(history['val_dice'], label='Validation', linewidth=2)
axes[1].axhline(y=best_dice, color='r', linestyle='--', label=f'Best: {best_dice:.4f}')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Dice Coefficient', fontsize=12)
axes[1].set_title('Dice Coefficient', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

# IoU
axes[2].plot(history['train_iou'], label='Train', linewidth=2)
axes[2].plot(history['val_iou'], label='Validation', linewidth=2)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('IoU Score', fontsize=12)
axes[2].set_title('IoU Score', fontsize=14, fontweight='bold')
axes[2].legend(fontsize=11)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['log_dir'], 'training_curves.png'), 
            dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTraining curves saved to: {os.path.join(CONFIG['log_dir'], 'training_curves.png')}")
print(f"All outputs saved to: {CONFIG['log_dir']}")

# Summary of all datasets
print(f"\n{'='*60}")
print("DATASET SUMMARY")
print(f"{'='*60}")
print(f"Train loader: {len(train_loader) if train_loader else 0} batches")
print(f"Validation loader: {len(val_loader) if val_loader else 0} batches")
print(f"Test loader: {len(test_loader) if test_loader else 0} batches")
print(f"Total datasets available: {sum([1 for loader in [train_loader, val_loader, test_loader] if loader is not None])}/3")
print(f"{'='*60}")

In [ ]:
pred_mask = inference(model, '/kaggle/input/monuseg2018/kmms_training/kmms_training/images/TCGA-21-5784-01Z-00-DX1.tif', device)
plt.imshow(pred_mask, cmap='gray')
plt.show()